# Cómo Craig Wright Engañó a Gavin Andresen

**Anatomía de una prueba falsa de Satoshi — y lo que nos enseña sobre ECDSA**

[← Volver al Esquema Didáctico de ECC](./00-ecc-teachable-scheme.ipynb) | [La Firma Sin Sentido (2018) →](./02-ecc-nonsense-signature.ipynb)

---

## El Contexto

En mayo de 2016, Craig Wright afirmó ser Satoshi Nakamoto. Publicó una
"prueba" en su blog y realizó una demostración privada a Gavin Andresen
(entonces el mantenedor principal de Bitcoin Core), quien salió convencido.

Ambas pruebas eran falsas. Pero nos enseñan algo profundo sobre lo que
las firmas realmente demuestran — y lo que no.

```
Lo que una firma DEMUESTRA:      Lo que una firma NO demuestra:
─────────────────────────        ──────────────────────────────────
Este (r,s) es válido para        QUIÉN lo produjo
este hash de mensaje z           CUÁNDO fue producido
bajo esta clave pública P        Que el firmante está frente a ti
```

**La lección fundamental:** La verificación te dice que una firma es *válida*.
No te dice que la persona que te la muestra la *creó*.

---

## Parte 1: El Truco del Blog (El Juego de Manos)

Wright publicó un post en su blog afirmando que firmó un texto de Jean-Paul Sartre
usando la clave privada de Satoshi. Así es como funcionó realmente el truco.

### La Forma Honesta de Demostrar Identidad

```
1. Alguien te da un mensaje FRESCO e IMPREDECIBLE
   → "Firma esto: 'El clima en Londres el 2 de mayo de 2016 es lluvioso'"

2. Lo firmas con tu clave privada
   → sig = ECDSA_sign(message, private_key)

3. Verifican usando la clave pública conocida
   → ECDSA_verify(message, sig, satoshi_pubkey) == True

4. Como solo el poseedor de la clave privada puede producir firmas
   válidas para mensajes frescos, esto demuestra identidad ✓
```

### Lo Que Wright Realmente Hizo

```
1. Tomó una transacción real de Satoshi de 2009 (pública en la blockchain)
   → tx 828ef3b0... (Satoshi enviando 10 BTC en enero de 2009)

2. Extrajo la firma de esa transacción
   → Esta firma ya es válida bajo la clave de Satoshi
   → CUALQUIERA puede leerla — es información pública

3. La presentó como si la acabara de crear para un texto de Sartre
   → "¡Mira, firmé este archivo de Sartre con la clave de Satoshi!"

4. La verificación pasa — porque es una firma REAL
   → Pero él no la creó. La copió.
```

### El Puente SHA256

La parte ingeniosa fue hacer que la firma de la transacción antigua *pareciera*
una firma de un archivo nuevo. Así es como:

```
Bitcoin firma transacciones así:
  z = SHA256(SHA256(modified_transaction))
  sig = ECDSA_sign(z, private_key)

OpenSSL verifica archivos así:
  z = SHA256(file_contents)
  ECDSA_verify(z, sig, pubkey)

El truco de Wright:
  El "archivo de Sartre" ERA el SHA256(modified_transaction)
  Entonces OpenSSL calcula: SHA256("Sartre") = SHA256(SHA256(modtx))
  ¡Que es exactamente z de la transacción original de Bitcoin!

  Mismo z → misma firma válida → la verificación pasa
```

In [ ]:
import hashlib

# El hash real de la transacción de Satoshi de 2009 (tx 828ef3b0...)
# Esto es SHA256 de la "transacción modificada" (signature_form)
modtx_hash_hex = "479f9dff0155c045da78402177855fdb4f0f396dc0d2c24f7376dd56e2e68b05"
modtx_hash = bytes.fromhex(modtx_hash_hex)

# Bitcoin calcula z como: SHA256(SHA256(modified_transaction))
# El "archivo de Sartre" de Wright contenía los bytes crudos de SHA256(modified_transaction)
# Entonces cuando OpenSSL hace SHA256(Sartre_file), obtiene SHA256(SHA256(modtx)) = z

sartre_is_modtx_hash = modtx_hash  # El "archivo de Sartre" es literalmente este hash

# Lo que OpenSSL calcula al "verificar" el archivo de Sartre:
z_openssl = hashlib.sha256(sartre_is_modtx_hash).hexdigest()

# Lo que Bitcoin calculó para la transacción original de 2009:
z_bitcoin = hashlib.sha256(modtx_hash).hexdigest()

print("=== El Puente SHA256 ===")
print(f"\nContenido del 'archivo de Sartre' (hex):")
print(f"  {modtx_hash_hex}")
print(f"\nLa verificación de OpenSSL calcula z = SHA256('archivo de Sartre'):")
print(f"  {z_openssl}")
print(f"\nEl z original de Bitcoin = SHA256(SHA256(modified_tx)):")
print(f"  {z_bitcoin}")
print(f"\n¿Mismo z? {z_openssl == z_bitcoin}")
print(f"\nComo z es el mismo, la firma de la transacción original")
print(f"también es válida como 'firma del archivo de Sartre'.")
print(f"Pero Wright nunca tocó una clave privada. Solo copió.")

### Reproduzcamos el Truco

Simularemos exactamente lo que hizo Wright, usando nuestras propias claves en lugar de las de Satoshi.
Esto demuestra que **cualquiera** puede hacerlo — no se necesita clave privada.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets

from ecc import *

def ecdsa_sign_z(z: int, d: int) -> tuple:
    while True:
        k = secrets.randbelow(SECP_N - 1) + 1
        R = scalar_mult(k, G)
        r = R.x % SECP_N
        if r == 0: continue
        s = (pow(k, SECP_N - 2, SECP_N) * (z + r * d)) % SECP_N
        if s == 0: continue
        return (r, s)

def ecdsa_verify_z(z: int, sig: tuple, pub: Point) -> bool:
    r, s = sig
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u1 = (z * s_inv) % SECP_N
    u2 = (r * s_inv) % SECP_N
    R = point_add(scalar_mult(u1, G), scalar_mult(u2, pub))
    return R.x % SECP_N == r

print("Primitivas criptográficas cargadas. ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  SIMULACIÓN: El Truco de Wright
# ═══════════════════════════════════════════════════════════════

# Paso 1: "Satoshi" crea una transacción real y la firma
print("PASO 1: El verdadero Satoshi firma una transacción en 2009")
print("=" * 55)

satoshi_private = secrets.randbelow(SECP_N - 1) + 1
satoshi_public = scalar_mult(satoshi_private, G)

# Simular una transacción siendo firmada
# Bitcoin: z = SHA256(SHA256(modified_tx))
fake_transaction = b"tx: send 10 BTC from Satoshi to Hal Finney, Jan 2009"
inner_hash = hashlib.sha256(fake_transaction).digest()  # SHA256(modtx)
z_original = int.from_bytes(hashlib.sha256(inner_hash).digest(), 'big')  # SHA256(SHA256(modtx))

original_sig = ecdsa_sign_z(z_original, satoshi_private)
r_orig, s_orig = original_sig

print(f"  Clave pública de Satoshi: ({hex(satoshi_public.x)[:18]}...)")
print(f"  Transacción firmada. Firma (r, s) registrada en la blockchain.")
print(f"  r = {hex(r_orig)[:18]}...")
print(f"  s = {hex(s_orig)[:18]}...")
print(f"  z (hash del mensaje) = {hex(z_original)[:18]}...")
print(f"\n  Esta firma + transacción son PÚBLICAS en la blockchain.")
print(f"  Cualquiera en el mundo puede leerlas.")

In [ ]:
# Paso 2: Craig Wright (el atacante) lee la blockchain
print("\nPASO 2: El atacante lee la blockchain (2016)")
print("=" * 55)

# El atacante NO tiene la clave privada de Satoshi.
# Pero PUEDE leer la transacción y su firma de la blockchain pública.
stolen_sig = original_sig  # Copiada de la blockchain — no se necesita clave privada
stolen_inner_hash = inner_hash  # SHA256(modified_tx) — derivable de datos públicos de la tx

print(f"  El atacante lee la transacción de 2009 de la blockchain.")
print(f"  Copia la firma: (r, s)")
print(f"  Calcula SHA256(modified_tx) a partir de los datos públicos de la transacción.")
print(f"  inner_hash = {stolen_inner_hash.hex()[:32]}...")

In [ ]:
# Paso 3: El truco — hacer que la firma antigua parezca nueva
print("\nPASO 3: El truco")
print("=" * 55)

# El atacante crea un archivo cuyo contenido es SHA256(modified_tx)
# Lo llama "Sartre" y afirma que es un texto de Jean-Paul Sartre
sartre_file = stolen_inner_hash  # ESTE es el truco

print(f"  El atacante crea un archivo llamado 'Sartre'.")
print(f"  Afirma que contiene un texto de Sartre (muestra solo el primer 14% en pantalla).")
print(f"  Contenido real: bytes crudos de SHA256(modified_tx)")
print(f"")
print(f"  Ahora el atacante dice:")
print(f'  "Firmé este archivo de Sartre con la clave de Satoshi. ¡Verifícalo!"')
print(f"")
print(f"  Te dice que verifiques usando OpenSSL:")
print(f"    openssl dgst -verify pub.pem -signature sig.der Sartre")
print(f"")
print(f"  OpenSSL calcula: z = SHA256(file_contents)")
print(f"                     = SHA256(SHA256(modified_tx))")
print(f"                     = ¡el EXACTO MISMO z de la transacción de 2009!")

In [ ]:
# Paso 4: Verificación — pasa, pero no demuestra nada
print("\nPASO 4: Verificación")
print("=" * 55)

# Lo que OpenSSL hace: z = SHA256(sartre_file)
z_from_sartre = int.from_bytes(hashlib.sha256(sartre_file).digest(), 'big')

print(f"  z de la tx original de 2009: {hex(z_original)[:24]}...")
print(f"  z del 'archivo de Sartre':   {hex(z_from_sartre)[:24]}...")
print(f"  ¿Mismo z? {z_original == z_from_sartre}")
print(f"")

# Verificar la firma robada contra el z del "Sartre"
valid = ecdsa_verify_z(z_from_sartre, stolen_sig, satoshi_public)
print(f"  ECDSA verify(sartre_z, stolen_sig, satoshi_pubkey) = {valid}")
print(f"")
print(f"  La verificación PASA. ✓")
print(f"  Pero el atacante NUNCA tocó la clave privada.")
print(f"  Solo copió una firma de la blockchain")
print(f"  y la envolvió en una presentación engañosa.")

In [ ]:
# Paso 5: Cómo REALMENTE demostrar identidad (la forma correcta)
print("\nPASO 5: Cómo funciona la VERDADERA prueba de identidad")
print("=" * 55)

# El verificador elige un mensaje de desafío FRESCO e IMPREDECIBLE
challenge = f"Sign this: random nonce {secrets.token_hex(16)}, date 2016-05-02"
z_challenge = int.from_bytes(hashlib.sha256(challenge.encode()).digest(), 'big')

print(f"  El verificador crea un desafío fresco:")
print(f'  "{challenge}"')
print(f"")

# Solo el verdadero Satoshi puede firmar esto
real_sig = ecdsa_sign_z(z_challenge, satoshi_private)
valid_real = ecdsa_verify_z(z_challenge, real_sig, satoshi_public)
print(f"  El verdadero Satoshi lo firma: valid = {valid_real} ✓")

# El atacante NO puede firmarlo — no tiene la clave privada
attacker_private = secrets.randbelow(SECP_N - 1) + 1  # Clave incorrecta
fake_sig = ecdsa_sign_z(z_challenge, attacker_private)
valid_fake = ecdsa_verify_z(z_challenge, fake_sig, satoshi_public)
print(f"  El atacante intenta:          valid = {valid_fake} ✗")

# ¿Puede el atacante reusar la firma antigua? No — z diferente
valid_replay = ecdsa_verify_z(z_challenge, stolen_sig, satoshi_public)
print(f"  Repetir firma antigua:        valid = {valid_replay} ✗")
print(f"")
print(f"  Un desafío fresco derrota el truco porque:")
print(f"  - El atacante no puede producir (r,s) para un nuevo z sin la clave")
print(f"  - Las firmas antiguas no verifican contra nuevos hashes de mensaje")
print(f"  - El verificador controla el mensaje, no el demostrador")

---

## Parte 2: La Demostración Privada (Engañando a Andresen)

El truco del blog fue descubierto en cuestión de horas por la comunidad Bitcoin. Pero la
demostración privada engañó a Gavin Andresen durante días.

### Qué Ocurrió

Wright llevó a Andresen en avión a Londres y realizó una firma "en vivo" en una habitación de hotel.
Andresen llevó una memoria USB con Electrum (una billetera Bitcoin con verificación de firmas).
Wright usó un "portátil nuevo, sellado de fábrica".

### Cómo (Probablemente) Se Falsificó

```
Vector de ataque: software de verificación modificado

Opción A — USB manipulado:
  La computadora cercana de Wright escribió silenciosamente un Electrum
  modificado en la memoria USB de Andresen. La versión modificada muestra
  "Válido" para CUALQUIER firma. Es un cambio de una sola línea de código:

    # Código original de Electrum:
    if signature_valid:
        show("Signature is VALID")
    else:
        show("Signature is INVALID")

    # Modificado (versión de Wright):
    if True:  # ← siempre válido
        show("Signature is VALID")
    else:
        show("Signature is INVALID")

Opción B — Portátil pre-cargado:
  El portátil "sellado de fábrica" ya tenía software modificado.
  Los sellos de fábrica se pueden volver a aplicar fácilmente.

Opción C — Entorno controlado:
  Toda la demostración fue en la habitación de hotel de Wright.
  Él controlaba cada variable.
```

### Por Qué Andresen Era Vulnerable

Andresen es un programador brillante. Pero el truco no era criptográfico —
era **ingeniería social** combinada con **control del entorno**.

```
Lo que Andresen verificó:     Lo que DEBERÍA haber verificado:
─────────────────────────     ──────────────────────────────────
El software dijo "Válido"     ¿Era realmente Electrum el software?
El portátil parecía nuevo     ¿Estaba el portátil realmente limpio?
El USB era "suyo"             ¿Fue modificado el USB al insertarlo?
Wright parecía seguro         ¿Estaba la matemática realmente ejecutándose?
```

---

## Parte 3: La Lección de ECDSA

### Las firmas son pruebas de CONOCIMIENTO, no pruebas de IDENTIDAD

ECDSA demuestra: *"alguien que conoce la clave privada $d$ produjo este $(r, s)$
para el hash de mensaje $z$."*

NO demuestra: *"la persona que te muestra esta firma es esa persona."*

### La ecuación de verificación no se preocupa por el tiempo

$$R' = \frac{z}{s} \times G + \frac{r}{s} \times P$$

Esta ecuación funciona de manera idéntica sin importar si la firma fue:
- Creada hace 5 segundos por la persona frente a ti
- Creada hace 7 años por alguien más y copiada de un registro público

### La defensa: Desafío-Respuesta

```
Protocolo correcto de prueba de identidad:

  Verificador                      Demostrador
  ───────────                      ───────────
  1. Generar nonce aleatorio
  2. Enviar mensaje de desafío ──→
                                   3. Firmar con clave privada
                              ←──  4. Enviar firma
  5. Verificar firma
  6. Aceptar SOLO si:
     - La firma es válida
     - El mensaje coincide con TU desafío
     - El desafío era impredecible
     - TÚ controlaste el software
```

### Propiedades que derrotaron el truco

| Propiedad | Por qué importa |
|-----------|-----------------|
| **Frescura** | El desafío debe ser nuevo — las firmas antiguas no se pueden repetir |
| **Impredecibilidad** | El demostrador no puede pre-calcular una firma válida |
| **Control del verificador** | El verificador debe controlar el software y el entorno |
| **Transparencia** | El mensaje completo y la firma deben ser inspeccionables |

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  EJERCICIO: Sé el Verificador
# ═══════════════════════════════════════════════════════════════
#
#  A continuación, alguien afirma ser "Satoshi" y te da una clave
#  pública y una firma. Tu tarea: determinar si esta es una prueba
#  legítima o una repetición al estilo Wright.
#
#  Pista: Verifica si la firma corresponde a TU desafío,
#  no a algún mensaje que el "demostrador" eligió.

# La clave pública de "Satoshi" (esta es la clave que estamos probando)
test_priv = secrets.randbelow(SECP_N - 1) + 1
test_pub = scalar_mult(test_priv, G)

# Una firma antigua de la "blockchain" (cualquiera puede verla)
old_message = b"Send 50 BTC to pizza guy"
old_z = int.from_bytes(hashlib.sha256(old_message).digest(), 'big')
old_sig = ecdsa_sign_z(old_z, test_priv)

# TU desafío fresco
your_challenge = f"Prove identity: nonce={secrets.token_hex(32)}"
your_z = int.from_bytes(hashlib.sha256(your_challenge.encode()).digest(), 'big')

# El demostrador te da una firma. ¿Pero para qué mensaje fue?
prover_sig = old_sig  # ← ¡El demostrador está repitiendo la firma antigua!

# TU verificación:
valid_for_old = ecdsa_verify_z(old_z, prover_sig, test_pub)
valid_for_challenge = ecdsa_verify_z(your_z, prover_sig, test_pub)

print("=== Análisis del Verificador ===")
print(f"\nTu desafío: {your_challenge[:50]}...")
print(f"\n¿La firma verifica?")
print(f"  Contra el mensaje antiguo: {valid_for_old}  (¡pero esto no demuestra nada!)")
print(f"  Contra TU desafío:         {valid_for_challenge}  (esto es lo que importa)")
print(f"\nVeredicto: {'¡FRAUDE — firma repetida!' if not valid_for_challenge else 'LEGÍTIMO'}")
print(f"\nEl demostrador te dio una firma válida — pero no para TU mensaje.")
print(f"El clásico truco de Wright.")

---

## Cronología

```
2009-01     Satoshi mina los primeros bloques, firma transacciones
            (las firmas quedan en la blockchain para siempre)

2016-04     Wright contacta a Andresen, BBC, The Economist
2016-05-02  Wright publica la "prueba" en su blog (truco del archivo de Sartre)
2016-05-02  Dan Kaminsky, ryanc, Reddit exponen el truco en HORAS
2016-05-02  Andresen publica "Creo que Craig Wright es Satoshi"
2016-05-05  Wright promete más pruebas, luego borra el post del blog
2016-05-06  Se revoca el acceso de commit de Andresen a Bitcoin Core

2024-03     Tribunal del Reino Unido dictamina: Wright NO es Satoshi (COPA v Wright)
```

## Puntos Clave

1. **Verificación ≠ Autenticación.** Una firma válida demuestra que la matemática funciona, no quién hizo la matemática.

2. **Las firmas públicas se pueden repetir.** Cada firma de Bitcoin es pública. Cualquiera puede presentar una como "suya".

3. **El desafío-respuesta es esencial.** El verificador debe controlar el mensaje que se firma.

4. **Controla el entorno.** Si el demostrador controla el software, la demostración no vale nada.

5. **Las afirmaciones extraordinarias necesitan pruebas transparentes.** Un verdadero Satoshi simplemente firmaría un mensaje fresco y lo publicaría para que todos lo verifiquen de forma independiente.

---

*Fuentes:*
- *[Recreating Craig Wright's Sartre File](https://rya.nc/sartre.html) — ryanc*
- *[Satoshi: how Craig Wright's deception worked](https://blog.erratasec.com/2016/05/satoshi-how-craig-wrights-deception.html) — Robert Graham, Errata Security*
- *[How Gavin Andresen was duped](https://www.cryptologie.net/article/350/how-gavin-andresen-was-duped-into-believing-wright-is-satoshi/) — David Wong*
- *COPA v Wright [2024] EWHC 1198*